In [ ]:
# Import required libraries
import os
import pandas as pd
import dash
from dash import html, dcc
from dash.dependencies import Input, Output
import plotly.express as px

# -----------------------------
# Step 1: Download CSV if missing
# -----------------------------
csv_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_dash.csv"
csv_file = "spacex_launch_dash.csv"

if not os.path.isfile(csv_file):
    print("Downloading CSV...")
    os.system(f'wget "{csv_url}" -O {csv_file}')

# -----------------------------
# Step 2: Read CSV
# -----------------------------
spacex_df = pd.read_csv(csv_file)
max_payload = spacex_df['Payload Mass (kg)'].max()
min_payload = spacex_df['Payload Mass (kg)'].min()

# -----------------------------
# Step 3: Create Dash app
# -----------------------------
app = dash.Dash(__name__)

# -----------------------------
# Step 4: App Layout
# -----------------------------
app.layout = html.Div(children=[

    html.H1(
        '🚀 SpaceX Launch Dashboard',
        style={'textAlign': 'center', 'color': '#503D36', 'font-size': 40}
    ),

    html.Br(),

    # TASK 1: Dropdown for Launch Site selection
    dcc.Dropdown(
        id='site-dropdown',
        options=[{'label': 'All Sites', 'value': 'ALL'}] +
                [{'label': site, 'value': site} for site in spacex_df['Launch Site'].unique()],
        value='ALL',
        placeholder="Select a Launch Site",
        searchable=True
    ),

    html.Br(),

    # TASK 2: Pie Chart
    html.Div(dcc.Graph(id='success-pie-chart')),

    html.Br(),

    html.P("Payload range (Kg):"),

    # TASK 3: Payload Range Slider
    dcc.RangeSlider(
        id='payload-slider',
        min=min_payload,
        max=max_payload,
        step=1000,
        marks={int(min_payload): str(int(min_payload)),
               int(max_payload): str(int(max_payload))},
        value=[min_payload, max_payload]
    ),

    html.Br(),

    # TASK 4: Scatter Plot
    html.Div(dcc.Graph(id='success-payload-scatter-chart'))
])

# -----------------------------
# Step 5: Pie Chart Callback
# -----------------------------
@app.callback(
    Output('success-pie-chart', 'figure'),
    Input('site-dropdown', 'value')
)
def update_pie_chart(selected_site):
    if selected_site == 'ALL':
        df = spacex_df[spacex_df['class'] == 1]
        fig = px.pie(df,
                     names='Launch Site',
                     title='Total Successful Launches by Site')
        return fig
    else:
        df = spacex_df[spacex_df['Launch Site'] == selected_site]
        fig = px.pie(df,
                     names=df['class'].map({1: 'Success', 0: 'Failure'}),
                     title=f'Success vs Failure for {selected_site}')
        return fig

# -----------------------------
# Step 6: Scatter Plot Callback
# -----------------------------
@app.callback(
    Output('success-payload-scatter-chart', 'figure'),
    [Input('site-dropdown', 'value'),
     Input('payload-slider', 'value')]
)
def update_scatter(selected_site, payload_range):
    low, high = payload_range
    filtered_df = spacex_df[(spacex_df['Payload Mass (kg)'] >= low) &
                            (spacex_df['Payload Mass (kg)'] <= high)]
    if selected_site == 'ALL':
        fig = px.scatter(filtered_df,
                         x='Payload Mass (kg)',
                         y='class',
                         color='Booster Version Category',
                         title='Payload vs Launch Outcome for All Sites',
                         hover_data=['Launch Site', 'Payload Mass (kg)'])
        return fig
    else:
        site_df = filtered_df[filtered_df['Launch Site'] == selected_site]
        fig = px.scatter(site_df,
                         x='Payload Mass (kg)',
                         y='class',
                         color='Booster Version Category',
                         title=f'Payload vs Launch Outcome for {selected_site}',
                         hover_data=['Launch Site', 'Payload Mass (kg)'])
        return fig

# -----------------------------
# Step 7: Run App on IBM IDE
# -----------------------------
if __name__ == '__main__':
    # Use host 0.0.0.0 and port 8080 for IBM IDE compatibility
    app.run(debug=True, host='0.0.0.0', port=8080)

ModuleNotFoundError: No module named 'pandas'